In [1]:
import torch

In [2]:
torch.__version__

'2.13.0+cu130'

In [3]:
import torch
import torch.nn as nn

In [5]:
class Encoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()  #상속 부모의 생성자 호출
        self.emb = nn.Embedding(vocab_size, 32) #임베딩
        self.gru = nn.GRU(32, 64, batch_first=True,bidirectional=True) #GRU
    def forward(self, x):
        return self.gru(self.emb(x))[0]

# 어텐션

In [ ]:
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.wq = nn.Linear(64, 64)         # q 가중치, 디코더 상태변환
        self.wk = nn.Linear(2 * 64 * 64)    # 인코더 hidden 변환
        self.v = nn.Linear(64, 1)           # 점수
        
    def forward(self, dec_h, enc_out):
        q = self.wq(dec_h).unsqueeze(1)     # (B, 1, H)
        k = self.wk(enc_out)                # (B, S, H)
        score = self.v(torch.tanh(q + k)).squeeze(-1)
        weights = torch.softmax(score, dim=1)
        context = (weights.unsqueeze(-1) * enc_out).sum(1)
        return context, weights

# 디코더

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size):
        super()._init__()
        self.emb = nn.Embedding(vocab_size, 32) # 임베딩
        self.attn = Attention()                 #어텐션
        self.cell = nn.GRUCell(32 + 2 * 64, 64) # GRU 한글자씩 -> GRUCell
        self.out = nn.Linear(64, vocab_size)

    def forward(self, y_prev, enc_out, h, uniform_len=None):
        # uniform_len == None 어텐션 끈다.
        if uniform_len is None:
            context, weights = self.attn(enc_out)
        else:
            weights = torch.zeros(enc_out.size(0), enc_out.size(1))
            weights[:, :uniform_len] = 1.0 / uniform_len
            context = (weights.unsqueeze(-1) * enc_out).sum(1)
        h = self.cell(torch.cat([self.emb(y_prev), context], dim=1), h)

        return self.out(h), h, weights